## Exploring Experiment Results
This notebook allows for exploration of experiment results.

In [103]:
import pandas as pd
from sklearn.metrics import classification_report

In [104]:
# Experiment we want to explore - customise these
TARGET_LABEL = "object"
EXPERIMENT_NUMBER = 10
# Use 0 to display every row, or a 1-based row number within the experiment.
ROW_NUMBER = 1

In [105]:
# Generated constants
FOLDER_NAME = f"{TARGET_LABEL}_classify"
RESULTS_PATH = f"../results/{FOLDER_NAME}"
EXPERIMENTS_DF_PATH = f"{RESULTS_PATH}/experiments.parquet"

In [106]:
def get_experiment_rows(df, experiment_number, row_number=0):
    """
    Return the requested 1-based row(s) within an experiment.

    Args:
        df (pd.DataFrame): Experiment results DataFrame.
        experiment_number (int): The experiment number to select.
        row_number (int): A 1-based row within that experiment, or 0 for all rows.

    Returns:
        pd.DataFrame: The selected row as a one-row DataFrame, or every row for the
            experiment when row_number is 0.

    Raises:
        ValueError: If the experiment does not exist, row_number is negative,
            or the requested row is outside the experiment.
    """

    # Filter the DataFrame for the specified experiment number
    df_experiment = df[df["experiment_number"] == experiment_number]

    # Catch errors for invalid experiment numbers or row numbers
    if df_experiment.empty:
        raise ValueError(f"Experiment number {experiment_number} not found in DataFrame.")
    if row_number < 0:
        raise ValueError("ROW_NUMBER must be 0 or a positive integer.")
    if row_number > len(df_experiment):
        raise ValueError(
            f"Row {row_number} not found in experiment {experiment_number}; "
            f"it contains {len(df_experiment)} row(s)."
        )
    # Return all rows if row_number is 0 or one row if a valid row_number is provided
    if row_number == 0:
        return df_experiment
    return df_experiment.iloc[[row_number - 1]]

In [107]:
def print_experiment_info(df, experiment_number, row_number=0):
    """
    Print the model and data configurations, training history, and evaluation results for a
    given experiment number and optional 1-based row number. A row number of 0
    prints every row.

    Args:
        df (pd.DataFrame): Experiment results DataFrame.
        experiment_number (int): The experiment number to select.
        row_number (int): A 1-based row within that experiment, or 0 for all rows.

    Raises:
        ValueError: If the experiment does not exist, row_number is negative,
            or the requested row is outside the experiment.
    """

    print(f"Experiment Number: {experiment_number}")

    # Get the requested rows for the experiment
    df_experiment = get_experiment_rows(df, experiment_number, row_number)

    print(f"Experiment Name: {df_experiment.iloc[0]['experiment_name']}\n")

    # Assign the first row number for display purposes. If row_number is 0, we start from 1.
    first_row_number = 1 if row_number == 0 else row_number

    # Print the information for each desired row in the experiment
    for display_row_number, (_, row) in enumerate(
        df_experiment.iterrows(), start=first_row_number
    ):
        # Print model and data configurations
        print(f"\n---- Row {display_row_number}: {row['run_name']} ----\n")
        print("Train Config:")
        for key, value in row["train_config"].items():
            print(f"  {key}: {value}")
        print("\nData Config:")
        for key, value in row["data_config"].items():
            print(f"  {key}: {value}")

        # Print training history
        history = row["history"]
        print("Training History:")
        for epoch_idx, metrics in enumerate(history):
            print(f"  Epoch {epoch_idx + 1}:")
            for metric_name, metric_value in metrics.items():
                print(f"    {metric_name}: {metric_value}")

        # Print evaluation results
        print("\nStandard Test Results:")
        print(f"  Test Accuracy: {row['test_acc']:.4f}")
        print(f"  Test Loss: {row['test_loss']:.4f}")
        print(f"  Weighted F1 Average: {row['test_weighted_f1_avg']:.4f}")

        print("\nUnseen Matched Test Results:")
        print(f"  Test Accuracy: {row['test_unseen_matched_acc']:.4f}")
        print(f"  Test Loss: {row['test_unseen_matched_loss']:.4f}")
        print(f"  Weighted F1 Average: {row['test_unseen_matched_weighted_f1_avg']:.4f}")

        print("\nUnseen Related Test Results:")
        print(f"  Test Accuracy: {row['test_unseen_related_acc']:.4f}")
        print(f"  Test Loss: {row['test_unseen_related_loss']:.4f}")
        print(f"  Weighted F1 Average: {row['test_unseen_related_weighted_f1_avg']:.4f}")

In [108]:
def print_classification_report(df, experiment_number, row_number=0, split="test"):
    """
    Print per-label classification reports for the selected evaluation split.

    Args:
        df (pd.DataFrame): Experiment results DataFrame.
        experiment_number (int): The experiment number to select.
        row_number (int): A 1-based row within that experiment, or 0 for all rows.
        split (str): One of "test", "test_unseen_matched", or "test_unseen_related".

    Raises:
        ValueError: If the experiment does not exist, row_number is negative,
            or the requested row is outside the experiment.
    """

    print(f"Classification Report for Experiment Number: {experiment_number}")
    print(f"Split: {split}")

    # Get the requested rows for the experiment
    df_experiment = get_experiment_rows(df, experiment_number, row_number)

    print(f"Experiment Name: {df_experiment.iloc[0]['experiment_name']}\n")

    # Assign the first row number for display purposes. If row_number is 0, we start from 1.
    first_row_number = 1 if row_number == 0 else row_number

    # Print the classification report for each desired row in the experiment
    for display_row_number, (_, row) in enumerate(
        df_experiment.iterrows(), start=first_row_number
    ):
        print(f"\n---- Row {display_row_number}: {row['run_name']} ----\n")

        if split == "test":
            y_true = row["test_y_true"]
            y_pred = row["test_y_pred"]
            label_ids = list(range(len(row["train_labels"])))
            label_names = [str(label) for label in row["train_labels"]]
        else:
            source_labels = pd.Series(row[f"{split}_y_true"])
            expected_labels = pd.Series(row[f"{split}_y_expected"])
            label_ids = []
            label_names = []
            for source_id, source_name in enumerate(row[f"{split}_labels"]):
                expected_ids = expected_labels[source_labels.eq(source_id)].unique()
                if len(expected_ids) != 1:
                    raise ValueError(
                        f"Unseen label {source_name!r} does not map to one expected label."
                    )
                label_ids.append(expected_ids[0])
                label_names.append(str(source_name))
            y_true = expected_labels
            y_pred = row[f"{split}_y_pred"]

        report = classification_report(
            y_true, y_pred, labels=label_ids, target_names=label_names, zero_division=0
        )
        print(report)

In [109]:
# Load the required experiment results from the Parquet file
df = pd.read_parquet(EXPERIMENTS_DF_PATH)
# df

In [110]:
print_experiment_info(df, EXPERIMENT_NUMBER, ROW_NUMBER)

Experiment Number: 10
Experiment Name: resnet18_candidate_seed_129_sweep


---- Row 1: pad_none ----

Train Config:
  checkpoint_dir: checkpoints/object_classify/010
  early_stopping_min_delta: 0.1
  early_stopping_patience: 3.0
  learning_rate: 2e-05
  min_learning_rate: 1e-06
  model_title: pad_none
  momentum: 0.9
  num_epochs: 25
  optimizer: adamw
  warmup_epochs: 1
  warmup_start_factor: 0.1
  weight_decay: 0.02

Data Config:
  batch_size: 32
  bg_path: data/baseline.jpg
  filtered_force_level: None
  filtered_motion: None
  norm_cache_path: configs/norm_cache.json
  norm_type: dataset
  num_workers: 4
  random_state: 129
  shuffle_map: {'test': False, 'train': True, 'val': False}
  split_size: 0.2
  stratify_label: object
  train_augmentations: {'color_jitter': None, 'horizontal_flip': None, 'random_resized_crop': False}
  transform_name: pad_224
Training History:
  Epoch 1:
    epoch: 1
    learning_rate: 2e-05
    train_acc: 76.2799834574028
    train_loss: 0.8505219928569059


In [111]:
print_classification_report(df, EXPERIMENT_NUMBER, ROW_NUMBER, split="test")

Classification Report for Experiment Number: 10
Split: test
Experiment Name: resnet18_candidate_seed_129_sweep


---- Row 1: pad_none ----

                precision    recall  f1-score   support

      football       0.99      1.00      0.99       960
        hammer       0.96      0.85      0.90       840
           mug       0.88      0.93      0.90       840
plastic_bottle       0.95      1.00      0.98       840
      scissors       0.87      0.96      0.91       840
sponge_scourer       1.00      1.00      1.00       840
     tea_towel       1.00      1.00      1.00       840
   tennis_ball       0.99      1.00      1.00       960
     tin_beans       0.89      0.70      0.78       840
   toilet_roll       0.97      0.88      0.93       840
    toothbrush       1.00      0.89      0.94       840
 tube_pringles       0.72      0.90      0.80       840
     tv_remote       1.00      1.00      1.00       960
  wooden_spoon       0.95      1.00      0.98       840

      accuracy    

In [112]:
print_classification_report(df, EXPERIMENT_NUMBER, ROW_NUMBER, split="test_unseen_matched")

Classification Report for Experiment Number: 10
Split: test_unseen_matched
Experiment Name: resnet18_candidate_seed_129_sweep


---- Row 1: pad_none ----

                precision    recall  f1-score   support

  media_remote       0.70      0.90      0.79      1440
 patterned_mug       0.84      0.73      0.78      2160
small_scissors       0.38      0.18      0.24      1440
      tin_peas       0.82      0.14      0.23      2160

     micro avg       0.72      0.47      0.57      7200
     macro avg       0.68      0.49      0.51      7200
  weighted avg       0.71      0.47      0.51      7200

